<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
.definition {border-left:5px solid #7c3aed;background:#f5f3ff;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## TW16 · Session 2 · Growth Rates, Frequency Changes, and Time-Aware Validation

        **Course level:** developing beginner  
        **Prior learning:** regular datetime indexes, shifts, rolling windows, and resampling  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        This session compares growth correctly across time, handles missing observations with restraint, rebases series, and protects temporal order during evaluation.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - calculate month-on-month, year-on-year, and compound annual growth
- distinguish upsampling from downsampling
- apply and justify limited interpolation
- rebase several measures to a common index
- create expanding and time-based train/test splits


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Retrieval | 10 min | Recall earlier Python and data skills |
| New concepts | 30 min | Learn with short, explained examples |
| Guided analysis | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation, predict the result, run the cell with <code>Shift + Enter</code>, then change one thing. Edit only cells marked <strong>Student task</strong> unless your lecturer says otherwise.</div>


### Load a regular monthly series

The three missing sales values remain visible for explicit treatment.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit

_candidates = [
    Path.cwd() / "data",
    Path.cwd() / "Python_Data_Science_TW09_TW17" / "data",
]
DATA_DIR = next((path for path in _candidates if path.exists()), _candidates[0])
if not DATA_DIR.exists():
    raise FileNotFoundError("Data folder not found. Start JupyterLab from the extracted course folder.")
print("Data folder:", DATA_DIR.resolve())

monthly = (
    pd.read_csv(DATA_DIR / "monthly_sales.csv", parse_dates=["month"])
    .sort_values("month")
    .set_index("month")
    .asfreq("MS")
)
print(monthly.head())
print("Missing sales:", monthly["sales"].isna().sum())


## 1. Growth rates answer different comparisons

- Month-on-month compares with the previous month.
- Year-on-year compares with the same month one year earlier.
- CAGR summarises constant compounded growth between start and end; it does not describe the path between them.


### Calculate monthly and annual growth

Missing observations remain missing rather than being filled silently.


In [ ]:
growth = monthly[["sales"]].copy()
growth["mom_pct"] = growth["sales"].pct_change(1, fill_method=None) * 100
growth["yoy_pct"] = growth["sales"].pct_change(12, fill_method=None) * 100

print(growth.tail(15).round(2))


        ### Calculate CAGR from valid endpoints

        The number of years is derived from the monthly distance between endpoints.

<div class="warning"><strong>Take care:</strong> CAGR is inappropriate when endpoints are non-positive and can be misleading when the path is volatile.</div>


In [ ]:
valid_sales = monthly["sales"].dropna()
start_value = valid_sales.iloc[0]
end_value = valid_sales.iloc[-1]
months_between = (
    (valid_sales.index[-1].year - valid_sales.index[0].year) * 12
    + valid_sales.index[-1].month - valid_sales.index[0].month
)
years_between = months_between / 12
cagr = (end_value / start_value) ** (1 / years_between) - 1

print("Start/end:", start_value, end_value)
print("Years:", round(years_between, 2))
print("CAGR:", f"{cagr:.2%}")


## 2. Missing time-series values

Interpolation estimates values between observations. It is an analytical assumption, not recovered truth. Keep an indicator, limit gap length, and test whether conclusions depend on the method.


### Apply time interpolation with a limit

Only short internal gaps are filled. The indicator preserves which values were estimated.


In [ ]:
completed = monthly.copy()
completed["sales_was_missing"] = completed["sales"].isna()
completed["sales_interpolated"] = completed["sales"].interpolate(
    method="time",
    limit=1,
    limit_area="inside",
)

print(completed.loc[completed["sales_was_missing"], ["sales", "sales_interpolated"]])


### Compare interpolation with a rolling alternative

Different assumptions produce different replacements; neither should be hidden.


In [ ]:
trailing_estimate = monthly["sales"].rolling(3, min_periods=2).mean()
comparison = pd.DataFrame({
    "observed": monthly["sales"],
    "time_interpolation": completed["sales_interpolated"],
    "trailing_mean": trailing_estimate,
})
print(comparison.loc[monthly["sales"].isna()].round(2))


## 3. Changing frequency

- Downsampling aggregates frequent data to a lower frequency.
- Upsampling creates a finer time grid and therefore introduces missing periods requiring an explicit rule.


### Downsample months to quarters

Sales and orders are flows summed within each quarter; marketing spend is averaged.


In [ ]:
quarterly = monthly.resample("QS").agg({
    "sales": "sum",
    "orders": "sum",
    "marketing_spend": "mean",
})
print(quarterly.head().round(2))


### Upsample a quarterly series

Forward fill repeats a known value and should only be used when that interpretation is valid. It is shown here transparently.


In [ ]:
quarterly_marketing = quarterly[["marketing_spend"]]
monthly_from_quarterly = quarterly_marketing.resample("MS").asfreq()
monthly_from_quarterly["forward_filled"] = monthly_from_quarterly["marketing_spend"].ffill()

print(monthly_from_quarterly.head(8))


## 4. Rebase series for relative comparison

When series use different units, divide each by its first valid value and multiply by 100. An index of 120 means 20% above the base period, not a raw value of 120.


### Rebase sales, orders, and marketing

Each column uses its own first valid observation.


In [ ]:
rebased = monthly.copy()
for column in ["sales", "orders", "marketing_spend"]:
    first_valid = rebased[column].dropna().iloc[0]
    rebased[column + "_index"] = rebased[column] / first_valid * 100

print(rebased.filter(like="_index").head().round(1))


### Plot rebased growth paths

The common base supports relative growth comparison while hiding absolute scale.


In [ ]:
index_columns = ["sales_index", "orders_index", "marketing_spend_index"]
fig, ax = plt.subplots(figsize=(9, 4))
for column in index_columns:
    ax.plot(rebased.index, rebased[column], label=column.replace("_index", "").title())
ax.axhline(100, color="black", linewidth=1, alpha=0.5)
ax.set_title("Relative change from the first observation")
ax.set_ylabel("Index: first valid period = 100")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()
plt.close(fig)


## 5. Seasonal profiles

Group comparable months across years to inspect recurring patterns. This is descriptive evidence; formal forecasting requires a separate modelling and validation process.


### Calculate average sales by calendar month

The month names are ordered by calendar number rather than alphabetically.


In [ ]:
seasonal = monthly.assign(
    month_number=monthly.index.month,
    month_name=monthly.index.month_name().str[:3],
)
seasonal_profile = (
    seasonal.groupby(["month_number", "month_name"])["sales"]
    .agg(["count", "mean", "std"])
    .reset_index()
    .sort_values("month_number")
)
print(seasonal_profile.round(2))


## 6. Time-aware train/test splits

Random splitting leaks future patterns into training. A final holdout should occur after training in time. TimeSeriesSplit provides expanding training windows for cross-validation.


### Create a final chronological holdout

The last 12 months are reserved as a future-like test period.


In [ ]:
model_data = completed[["sales_interpolated", "orders", "marketing_spend"]].dropna()
train = model_data.iloc[:-12]
test = model_data.iloc[-12:]

print("Train:", train.index.min().date(), "to", train.index.max().date())
print("Test:", test.index.min().date(), "to", test.index.max().date())
assert train.index.max() < test.index.min()


### Inspect expanding cross-validation windows

Each validation fold occurs strictly after its corresponding training fold.


In [ ]:
splitter = TimeSeriesSplit(n_splits=4)
split_rows = []
for fold, (train_index, validation_index) in enumerate(splitter.split(train), start=1):
    split_rows.append({
        "fold": fold,
        "train_start": train.index[train_index[0]],
        "train_end": train.index[train_index[-1]],
        "validation_start": train.index[validation_index[0]],
        "validation_end": train.index[validation_index[-1]],
    })
print(pd.DataFrame(split_rows))


## Guided time-series briefing

A concise briefing combines recent MoM/YoY growth, long-run CAGR, missing-data treatment, seasonal pattern, and the date boundary used for evaluation.


### Assemble a briefing record

The record makes scope and assumptions explicit.


In [ ]:
latest_valid_growth = growth.dropna(subset=["sales"]).iloc[-1]
briefing = pd.Series({
    "data_start": monthly.index.min(),
    "data_end": monthly.index.max(),
    "observations": len(monthly),
    "interpolated_values": int(completed["sales_was_missing"].sum()),
    "latest_mom_pct": latest_valid_growth["mom_pct"],
    "latest_yoy_pct": latest_valid_growth["yoy_pct"],
    "cagr_pct": cagr * 100,
    "test_periods": len(test),
})
print(briefing.round(2))


## Student coding lab

State the comparison period and missing-data treatment next to every growth figure.


        ### Student task 1: Compare growth measures

        <div class="task"><strong>Your job:</strong> Calculate MoM and YoY growth for sales and orders, then report the latest valid values with exact comparison periods.</div>

        **Check your work**

        - Both measures use fill_method=None.
- Sales and orders are both included.
- The interpretation distinguishes one month from one year.


In [ ]:
# STUDENT TASK 1
task_growth = pd.DataFrame(index=monthly.index)
print(task_growth.tail())


        ### Student task 2: Calculate CAGR

        <div class="task"><strong>Your job:</strong> Write a reusable CAGR function that accepts start value, end value, and years; validate positive inputs and apply it to sales.</div>

        **Check your work**

        - Invalid non-positive inputs raise ValueError.
- Compounding uses 1/years.
- The actual data calculation derives elapsed years.


In [ ]:
# STUDENT TASK 2
def calculate_cagr(start_value, end_value, years):
    return None

task_cagr = calculate_cagr(100, 125, 2)
print(task_cagr)


        ### Student task 3: Audit interpolation

        <div class="task"><strong>Your job:</strong> Create linear, time, and trailing-mean alternatives for missing sales and compare only the affected months.</div>

        **Check your work**

        - Original values remain unchanged.
- At least two distinct assumptions are visible.
- A preferred choice is justified rather than treated as truth.


In [ ]:
# STUDENT TASK 3
interpolation_audit = pd.DataFrame({"observed": monthly["sales"]})
# Add three alternatives and filter missing original rows.
print(interpolation_audit.loc[monthly["sales"].isna()])


        ### Student task 4: Rebase and compare

        <div class="task"><strong>Your job:</strong> Rebase sales, orders, and marketing spend to 100 at their first valid values, assert each base equals 100, and plot.</div>

        **Check your work**

        - Each series has its own valid base.
- The axis is labelled as an index.
- A limitation notes that absolute scale is hidden.


In [ ]:
# STUDENT TASK 4
task_rebased = monthly.copy()
fig, ax = plt.subplots(figsize=(9, 4))
# Calculate, validate, and plot indices.
plt.close(fig)
print(task_rebased.head())


        ### Student task 5: Create time-aware evaluation windows

        <div class="task"><strong>Your job:</strong> Reserve the final 12 months as test data and make four expanding validation folds from earlier data. Prove all validation dates follow training dates.</div>

        **Check your work**

        - No random shuffle is used.
- The test set is last in time.
- Every fold passes a chronological assertion.



**Optional extension:** Add a gap between training and validation when operational delays could cause leakage.


In [ ]:
# STUDENT TASK 5
window_report = []
# Add holdout and TimeSeriesSplit logic.
print(pd.DataFrame(window_report))


        ## Knowledge check

        Answer these without running new code first.

        1. How does YoY growth differ from MoM growth?
2. What path information does CAGR hide?
3. Why preserve an interpolation indicator?
4. What does a rebased index of 120 mean?
5. Why is random splitting risky for time series?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** TW17 integrates the module into a group project, presentation, reproducibility review, and final technical checklist.
